# Exploring artifacts in your metaflow runs
This workbook serves to explore data artifacts in previous runs and pull artifacts for usage. Users are able to and promoted to leverage artifacts within their flows for a variety of use cases.

_For more information check the Metaflow docs [here](https://docs.metaflow.org/api/client) or [here](https://docs.metaflow.org/metaflow/client)_

This notebook is not exhaustive. If you would like to address a new exploration method please create a Github Issue for this repository or email One AI support.

Have some free time? Feel free to create a branch - add some more examples and create a PR. Our team would be happy to review your work and increase our tutorial content :)

## TOC:
* [Introduction](#intro)
* [Client API 101](#client-api)
* [Instantiating Objects](#artifacts)
    * [Accessing Flows](#flows)
    * [Accessing Runs](#runs)
    * [Accessing Steps](#steps)
    * [Accessing Tasks](#tasks)
* [Iterating Through Flows](#iterate)
* [Accessing Data Artifacts](#data-artifacts)
    * [Tracking Variable over Runs](#runs)
    * [Fetching Models](#models)
    * [Observing Input Files](#files)
* [Filtering by Tags](#tags)
* [Explore Namespaces](#namespaces)
* [Explore Production Namespaces](#production)

### Introduction <a class="anchor" id="intro"></a>
One of the great benefits of metaflow is that it aims to version everything. This means that Flows, runs, steps, tasks and data artifacts (e.g., variables, input files, models) are all stored in cloud. Simplifying your life of tracking experiments and coordinating projects amongst your peers.

Metaflow provides a client API that is used to inspect results of past runs. This notebook will demonstate some functionality of which this API is capable off.

![METAFLOWOBJECT](./images/metaflow_hierachy.png)
*Metaflow objects are organized in a hierachy*

### Client API 101 <a class="anchor" id="client-api"></a>
The details of the Client API offerings can be found [here](https://docs.metaflow.org/api/client).

Essentially, the Client API operations are filtered by namespace. By default, when a user uses the Client API in a notebook or a python script, the API only returns results that are tagged to the users namespace 'user:\<user_name\>'.

Namespaces are not about security or access control, but rather to keep runs organized. Jump to the _Exploring Namespaces_ section if you would like to learn more.

Tagging and namespaces, together with the Client API, are the main ways for accessing results of past runs. Metaflow uses these mechanisms to organize and isolate results automatically, so in most cases you don't have to do anything. 

Wow! Talk about peace of mind. Let us get our environment variables all set up so we can begin.

In [0]:
# Add parent directory to sys.path that specifies the path to look
# for packages and modules
import os, sys

sys.path.insert(1, os.path.abspath(".."))

In [0]:
# load environment variables defined in the .env file
from ml_services.utils.env_variables import Env

e = Env()

In [0]:
# install relevant libraries for this exploration
from metaflow import Metaflow, Flow, Run, Step, namespace

#### Instantiating Objects <a class="anchor" id="artifacts"></a>
You can instatiate a specific object at any level of the hierachy by providing a corresponding pathspec. A pathspec may be identified from Metaflow logs.

Every object has the following properties available:

- user_tags: user-defined tags assigned to the object's run
- system_tags: system-defined (immutable) tags assigned to the object's run
- tags: the union of user_tags and system_tags
- created_at: creation timestamp
- parent: parent object
- pathspec: object fully qualified name
- path_components: list containing the elements in pathspec

To take a look at the different tags attached to a Flow, skip to the filtering by tags section. For now, lets begin with basics:

In [0]:
"""
The root object, Metaflow is the entry point to all other objects.
You can list all flows that have been run in the past with the code below.
However, remember this is only listing flows pertaining to your namespace. Feel free to set namespace(None)
to mitigate this isolation.
"""
mf = Metaflow()
print(mf.flows)

##### Accessing Flows <a class="anchor" id="flows"></a>
Flows can be called via a pathspec

A Flow represents all existing flows with a certain name, in other words, classes derived from FlowSpec. A container of Run objects.

Flow has two additional properties related to determining the latest run for the flow. Note that any Run returned will be in the current namespace.

- latest_run: Run of the latest run whether or not it has completed or has been successful
- latest_successful_run: Run of the latest successful (and therefore completed) run.

In [0]:
print(Flow("IrisFlow").pathspec)
print(Flow("IrisFlow").latest_run)
print(Flow("IrisFlow").latest_successful_run)

##### Accessing Runs <a class="anchor" id="runs"></a>
A Run represents an execution of a Flow. It is a container of Steps.


Run also has a few additional properties to make it easy to access commonly used information:

- data: A quick way to access the data object of the end task of this run. In other words, this is the quickest way to access the data produced at the end of the flow.
- successful: A boolean indicating whether or not the run completed successfully. Note that this will return False if the run has not completed (ie: is still in progress).
- finished: A boolean indicating whether or not the run completed. The returned value will be True whether or not the run was successful.
- finished_at: A datetime object indicating the completion time of the run. This will be None if the run has not completed
- code: In certain circumstances, the code used for this run is saved and persisted; this allows you to access this code.
- end_task: A quick access to the Task object of the last step in the run.

In [0]:
latest_run = Run("IrisFlow/77")
print(latest_run)
print("is successful:", latest_run.successful)
print("data:")
print(latest_run.data)

##### Accessing Steps <a class="anchor" id="steps"></a>
A Step typically has a single Task. A Step will have multiple Task objects as its children if it is a foreach step; each Task will correspond to a single execution of the Step.

To access an iterator over the tasks of a step and filter by tags, use the tasks() method.

Step has a few additional properties as well:

- task: Convenience method to return the unique Task associated with this Step. If a Step has more than one Task, this will return any of them (no order guaranteed).
- finished_at: A datetime object indicating the completion time of the step. A step is complete when all its tasks are complete.
- environment_info: A dict object containing metadata for the execution environment.

In [0]:
print("steps in latest run")
print(list(latest_run.steps()))
print("investigate the tasks in cross validation")
print(list(Step("IrisFlow/77/cross_validation").tasks()))

##### Accessing Tasks <a class="anchor" id="tasks"></a>
A Task represents an execution of a Step.

It contains all DataArtifact objects produced by the task as well as metadata related to execution.

Note that the @retry decorator may cause multiple attempts of the task to be present. Usually you want the latest attempt, which is what instantiating a Task object returns by default. If you need to e.g. retrieve logs from a failed attempt, you can explicitly get information about a specific attempt by using the following syntax when creating a task:

Task('flow/run/step/task', attempt=<attempt>)

where attempt=0 corresponds to the first attempt etc.
    
Since a Task is the actual unit of execution in Metaflow, these objects contain the richest set of properties:

- data: A convenience method to access all data produced by this Task. 
- artifacts: A convenience method to access all DataArtifact objects produced by this Task. 
- successful: A boolean indicating whether or not this Task completed successfully.
- finished: A boolean indicating whether or not this Task completed.
- exception: If an exception was raised by this Task (ie: it did not complete successfully), it will be contained here.
- finished_at: A datetime object indicating the completion time of this Task.
- stdout: A string containing the standard output of this Task.
- stderr: A string containing the standard error of this Task.
- code: The code used to execute this Task, if available.
- environment_info: A dict object containing metadata for the execution environment.

In [0]:
print("lets get the predict task of the last run")
predict_task = Step("IrisFlow/77/predict").task
print("=======================")
print(predict_task.artifacts)
print("=======================")
print(predict_task.data)
print("=======================")
print(predict_task.data.accuracy)

#### Iterating Through Flows <a class="anchor" id="iterate"></a>
Each object is a container (an iterable) that can be used to iterate over objects that are below it in the hierachy.

In [0]:
"""
Iterating over list of Flows, yields a list of Runs
"""
for flow in mf.flows:
    print("Latest runs for:", flow)
    print(flow.latest_run)
    print("data in latest run:")
    print(flow.latest_run.data)
    print("\n=============\n")

In [0]:
"""
Iterating over a list of Runs, yields a list of Steps
"""
iris_flow = Flow("IrisFlow")
latest_run = iris_flow.latest_run
print("Steps in latest run:")
for step in latest_run:
    print(step)

#### Accessing Data Artifacts <a class="anchor" id="data-artifacts"></a>
One of the most typical use cases of the client API is to access data artifacts produced by runs. Each data artifact is represented by a DataArtifact object whose parent is a Task.

DataArtifact is a container object for the actual value. Besides the value, DataArtifact includes metadata about the artifact, such as its time of creation.

Often you are only interested in the value of an artifact. For this typical use case we provide a convenience property .data in the Task object. The .data property returns a container which has all artifacts produced by the task as attributes.

In [0]:
print("lets get the predict task of the last run")
predict_task = Step("IrisFlow/77/predict").task
print("\nhere are the artifacts")
print(predict_task.artifacts)
print("\nsome additional metadata is included in the artifact like finished_at")
print(list(predict_task)[0].finished_at)

##### Tracking Variable over Runs <a class="anchor" id="runs"></a>

In [0]:
# lets track the accuracy over all runs within IrisFlow
runs = Flow("IrisFlow")
for run in runs:
    steps = [step.path_components[-1] for step in run]
    if "predict" in steps:
        if run.data.accuracy:
            print("run %s, accuracy: %f" % (run.pathspec, run.data.accuracy))

##### Accessing model for prediction <a class="anchor" id="models"></a>
See [here](https://outerbounds.com/docs/use-keras-with-metaflow) for how to save and load models that are not stored via pickle file.

In [0]:
# lets get a model
model = Flow("IrisFlow").latest_run.data.predict_model
import pandas as pd

df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data",
    names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"],
    header=None,
)
x = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
# y = df.species
print(model.predict(x)[0:5])

##### Observing an Input File <a class="anchor" id="files"></a>


In [0]:
file_path = Flow("IrisFlow").latest_run.data.config_file
print(file_path)

### Filtering by tags <a class="anchor" id="tags"></a>
See [here](https://docs.metaflow.org/metaflow/tagging#filtering-by-tags) for more information.

Essentially, every run has a set of tags attached, that is, user-defined annotations.

In [0]:
# you can see system tags like this
print(Run("IrisFlow/77").system_tags)
# you can see all tags like this
print(Run("IrisFlow/77").tags)
# we can get runs with only a specific tag
print(list(Flow("IrisFlow").runs("project_branch:prod")))

### Exploring Namespaces <a class="anchor" id="namespaces"></a>

You can freely explore results produced by other people by using the namespace command. See more [here](https://docs.metaflow.org/metaflow/tagging)

In [0]:
from metaflow import namespace

namespace("user:utkarsh")
run = Flow("IrisFlow").latest_successful_run
print(run.data)

Set the namespace to None to see all flows across namespaces. We do not recommend this approach for post analysis because user runs would then become hard to sort through.

In [0]:
namespace(None)
mf = Metaflow()
print(mf.flows)

### Exploring Production Namespaces <a class="anchor" id="production"></a>

You can see a list of factory flows by setting the namespace to runtime:argo-workflows. See below. We do not recommend this approach because generally you want to focus on your project within factory, and not all projects (it can get a bit messy just like the above example)

In [0]:
namespace("runtime:argo-workflows")
mf = Metaflow()
print(mf.flows)

Now, you can imagine the amount of flows you would see in Factory would increase as the number of projects are able to transition to production. We advise using the "production:<project_name>" namespace when accessing factory results for your flows.

Lets take a look:

In [0]:
# lets first look at an example of factory tags that are produced
print(Run("MetaFlowIris/argo-metaflowiris.prod.metaflowiris-1663106400").tags)

In [0]:
# lets use the production tag to switch our namespace:
namespace("production:metaflow_iris")
mf = Metaflow()
print(mf.flows)
print("accessing data artifacts from a production run to observe results")
run = Flow("MetaFlowIris").latest_successful_run
print(run.data.accuracy)

And there you have it! If you have any questions, please contact the liveops support team for One AI and engage with the Metaflow community 